In [1]:
import ast
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
#from Functions.AutoCloud_V4_4 import *
from Functions.AutoCloud_V4_6 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
from Functions.TedaOptimize_V4_6 import *     # Optimize, OptimizeSeq, BestParams, SimSeq
import copy

FileName = ipynbname.name()

RS = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')
sig = HI['PC1'].values
PARAMS_PATH = r'Optimization2\TedaRNN_Opt_R2_TESTE_B1_1-B1_2_adapt_symmetric_G0P0\mono\Optimization.csv'
dfP = pd.read_csv(PARAMS_PATH).iloc[:,:11]
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values
EOL = 0.3
W = 1

c:\Users\Usuario\Desktop\Python\Batteries_Prognostics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = r'Optimization\TedaRNN_Opt_R4_SEQ1_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
df = pd.read_csv(path)
df = df.sort_values(df.columns[0], ascending=True)
df

,WAPE_mean,m,nI,nR,nO,mO,N1,N2,N3,TAU,activation,WAPE_RUL_B1_1,WAPE_HI_B1_1,WAPE_RUL_B1_2,WAPE_HI_B1_2
4,0.135473,1.9875,40,[75],1,29,1,9,8,21,0,0.114777,0.007402,0.132137,0.016630
14,0.136852,2.0125,20,"[65, 21, 70]",3,18,1,2,9,25,0,0.101048,0.021432,0.146930,0.004292
9,0.137215,2.0125,38,"[16, 54]",9,0,1,2,7,20,0,0.111611,0.028971,0.130478,0.003370
3,0.137306,1.9875,40,[75],2,29,1,8,7,21,0,0.119329,0.007394,0.131264,0.016625
13,0.144633,2.0250,21,"[71, 24, 68]",3,18,1,4,9,25,0,0.128379,0.027783,0.128862,0.004242
19,0.147892,2.0250,23,"[21, 54, 63, 65]",4,1,1,2,5,11,0,0.145123,0.011083,0.108260,0.031317
2,0.149006,1.9875,40,[75],3,29,1,7,8,20,0,0.126842,0.007449,0.140444,0.023276
8,0.155089,2.0500,36,"[20, 51]",9,2,1,1,1,17,0,0.145401,0.013640,0.149310,0.001827
1,0.158069,2.0000,40,[75],2,29,1,9,4,20,0,0.127271,0.007455,0.142025,0.039386
18,0.159436,1.7500,9,"[57, 57, 38, 51]",5,1,1,2,8,21,0,0.156289,0.021610,0.129744,0.011229


In [ ]:

m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X,Y,Z,'single')
teda.HI_Granularity()
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

In [ ]:
df = Optimize(RS=RS, HI=HI['PC1'], FileName=FileName, OptSampler='tpe', OptSeed=1,
              n_study=1, n_trials=100,patience=22, timeout=3600, nLS=[1,2], mS=[1.5,4.5])


In [ ]:
data = [(f'B{b}', pd.read_csv(fr'Dataset\DSIR1n\B{b}.csv'), pd.read_csv(fr'Dataset\HI1\Bearing{b}.csv')['PC1'])
        for b in ('1_1', '1_2')]
dfS = OptimizeSeq(data=data, FileName=FileName, OptSampler='random', OptSeed=1,OptPrune=True, agg='mean',
                  n_study=3, n_trials=1000, patience=5e2, timeout=3600, nLS=[1,3], mS=[1.75,2.05])
dfS = OptimizeSeq(data=data, FileName=FileName, OptSampler='tpe', OptSeed=1,OptPrune=True, agg='mean',
                  n_study=1, n_trials=1000, patience=5e2, timeout=3600, nLS=[1,3], mS=[1.75,2.05])


In [21]:
RS1 = pd.read_csv(r'Dataset\DSIR1n\B1_1.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_1.csv')

sig = HI['PC1'].values
PARAMS_PATH = r'Optimization\TedaRNN_Opt_R3_Seq_B1_1-B1_2_restart_symmetric_G0P0\mono\Optimization.csv'
dfP = pd.read_csv(PARAMS_PATH).iloc[:,:11]
params = dfP.sort_values(dfP.columns[0], ascending=True).iloc[0, 1:].values
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
RHO = 0.1       # rho estático: intervalo do HI = predição -/+ RHO * erro estimado (protocolo 3)
X, Y, Z, kX = PrepareData(RS1, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )

teda.SimGranularRUL(J,L,X,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")

PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

RS2 = pd.read_csv(r'Dataset\DSIR1n\B1_2.csv')
HI = pd.read_csv(r'Dataset\HI1\Bearing1_2.csv')
sig = HI['PC1'].values
X, Y, Z, kX = PrepareData(RS2, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX) 

teda.restart(ref,'keep',Y[0])
teda.SimGranularRUL(J,L,X,Y,Z,'single')
print(f"HI : cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f} | ")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide')

HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


HI : cov=0.000 | sp=1.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000 | 


In [20]:
brngs = [f'B1_{i+1}' for i in range(5)]
brngs = ['B1_1','B1_2','B1_3','B1_4','B1_5','B2_1','B2_2','B2_4',][:2]
paths = [f'Dataset\DSIR1n\{brng}.csv' for brng in brngs]
DSIs = [pd.read_csv(path) for path in paths]
#DSIs = [DSI.drop(["FTF"], axis=1) for DSI in DSIs]
ftrs = len(DSIs[0].columns)
DSIs = [RS1,RS2]
teda = AutoCloud(m=m,nI=nI,nR=nR,nO=nO,mO=mO,ηS=[N1,N2,N3],
			  tau=τ,eol=EOL,ref=ref,act=act,
			  wtaG=False,wtaP=False,store=True,feedback='symmetric',
			  rho=RHO,intr_state='carry'
			  )
w = 1
for DSI in DSIs: 
    teda.SimGranulation(WindowDSI(DSI,w))

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=ftrs,dsi_mode='hide')

HI (HI_Granularity): cov=0.000 | sp=0.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000
